<a href="https://colab.research.google.com/github/NSPFD/Qnickeel26/blob/main/Assign_GroverMaxCut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework - Grover MaxCut

The places where you have enter code are marked with `# YOUR CODE HERE`.

In [1]:
!pip install cirq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 52.3 MB/s eta 0:00:00


In [2]:
import cirq
from cirq import H, X, Y, Z, CX, inverse

## Question 1 (4 points)

Write a function, `oracle010`, that implements an oracle that marks the state $|010 \rangle$. The function `oracle010` has

* input: `qq`, a 3-qubit register
* returns: `None`

The function should append a sequence of gates to `qq` to mark the state $|010\rangle$ only. Don't append any measurements to `qq`.

To help you test the function, we have provided the `grover_diffusion` and `grover` functions.

In [3]:
import cirq

def oracle010(qq):
    # 1. Flip qubits where we expect a '0' to map |010> to |111>
    yield cirq.X(qq[0])
    yield cirq.X(qq[2])

    # 2. Apply a multi-controlled Z gate to invert the phase of |111>
    # We use qq[0] and qq[1] as controls, and qq[2] as the target of the Z gate
    yield cirq.Z(qq[2]).controlled_by(qq[0], qq[1])

    # 3. Uncompute the X gates to restore the register state
    yield cirq.X(qq[0])
    yield cirq.X(qq[2])

In [4]:
# visualize your implemented gates
qqTest = cirq.LineQubit.range(3)
circuit = cirq.Circuit()
circuit.append(oracle010(qqTest))
circuit

0: ───X───@───X───
          │
1: ───────@───────
          │
2: ───X───@───X───

In [5]:
# To check your solution, we need some to implement grover
def grover_diffusion(qq,n):
    yield H.on_each(*qq)
    yield X.on_each(*qq)
    yield Z(qq[n-1]).controlled_by(*(qq[0:n-1]))
    yield X.on_each(*qq)
    yield H.on_each(*qq)

In [6]:
def grover(trials_number):
    n=3
    qq = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()
    circuit.append(H.on_each(*qq))

    for i in range(2):
        circuit.append(oracle010(qq))
        circuit.append(grover_diffusion(qq,n))
    circuit.append(cirq.measure(*qq, key='result'))

    # determine the statistics of the measurements
    s = cirq.Simulator()
    samples = s.run(circuit, repetitions=trials_number)

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    print(counts)
    return counts.get('010')

In [7]:
# run grover to test if your function gives the right answer
grover(100)

Counter({'010': 94, '001': 3, '000': 2, '101': 1})


94

In [ ]:
# hidden tests in this cell will be used for grading.

## Question 2 (6 points)

Graph $G$ has 5 vertices and 6 edges: (0,3), (0,4), (1,3), (1,4), (2,3), (2,4).

Write an oracle for the graph $G$ to check whether it admits a valid 2-coloring.

The function `oracle2` has

* input: `qq`, a 12-qubit register
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits 0-4 for the vertices, 5-10 for the edges and 11 as the ancilla.

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation2` functions.

In [8]:
import cirq

def oracle2(qq):
    # Define the 6 edges given in the problem statement
    edges = [(0,3), (0,4), (1,3), (1,4), (2,3), (2,4)]

    # 1. Compute: Check each edge and store the result in qubits 5 to 10
    for i, (u, v) in enumerate(edges):
        edge_ancilla = 5 + i
        yield cirq.CX(qq[u], qq[edge_ancilla])
        yield cirq.CX(qq[v], qq[edge_ancilla])

    # 2. Mark: If all 6 edges are properly colored, flip the master ancilla (qubit 11)
    # qq[5:11] dynamically grabs qubits 5, 6, 7, 8, 9, and 10
    yield cirq.X(qq[11]).controlled_by(*qq[5:11])

    # 3. Uncompute: Reset the edge qubits back to state |0>
    for i, (u, v) in enumerate(edges):
        edge_ancilla = 5 + i
        yield cirq.CX(qq[v], qq[edge_ancilla])
        yield cirq.CX(qq[u], qq[edge_ancilla])

In [9]:
# We need some code so you can check your solution
def oracle_computation2(qq):
    yield oracle2(qq)
    yield Z(qq[11])
    yield inverse(oracle2(qq))

In [10]:
def grover2(trials_number):
    import cirq
    from cirq import X, H, Z, inverse, CX
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(12)
    n=5

    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation2(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [11]:
#You can use this cell to test your solution
shots=1000
grover2(shots)

Counter({'00011': 470,
         '01101': 6,
         '11100': 443,
         '10101': 4,
         '00100': 5,
         '11111': 4,
         '10000': 2,
         '10010': 5,
         '00101': 4,
         '01011': 6,
         '01100': 4,
         '01000': 5,
         '00001': 1,
         '11000': 1,
         '01111': 3,
         '00111': 3,
         '10111': 4,
         '01010': 1,
         '11101': 3,
         '10100': 1,
         '10011': 5,
         '10110': 2,
         '11010': 4,
         '00010': 3,
         '11110': 5,
         '00000': 1,
         '00110': 1,
         '10001': 3,
         '11001': 1})

In [ ]:
# hidden tests in this cell will be used for grading.

## Question 3 (10 points)

Graph $G$ has 4 vertices and 5 edges: (0,1), (0,2), (0,3), (1,2), (1,3)

Write an oracle for the graph $G$ to check whether there exists a coloring with at least 4 edges connecting vertices with different colors.

The function `oracle3` has

* input: `qq`, a 13-qubit register
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits
- 0-3 for the vertices,
- 4-8 for the edges,
- 9-11 for the addition (remember we need three qubits here for addition unlike the last question), and
- 12 as the ancilla.

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation3` functions.

In [16]:
import cirq

def oracle3(qq):
    # Define the 6 edges given in the problem statement
    edges = [(0,3), (0,4), (1,3), (1,4), (2,3), (2,4)]

    # 1. Compute: Check each edge and store the result in qubits 5 to 10
    for i, (u, v) in enumerate(edges):
        edge_ancilla = 5 + i
        yield cirq.CX(qq[u], qq[edge_ancilla])
        yield cirq.CX(qq[v], qq[edge_ancilla])

    # 2. Mark: If all 6 edges are properly colored, flip the master ancilla (qubit 11)
    # qq[5:11] dynamically grabs qubits 5, 6, 7, 8, 9, and 10
    yield cirq.X(qq[11]).controlled_by(*qq[5:11])

    # 3. Uncompute: Reset the edge qubits back to state |0>
    for i, (u, v) in enumerate(edges):
        edge_ancilla = 5 + i
        yield cirq.CX(qq[v], qq[edge_ancilla])
        yield cirq.CX(qq[u], qq[edge_ancilla])

In [17]:
# We need some code so you can check your solution
def oracle_computation3(qq):
    yield oracle3(qq)
    yield Z(qq[12])
    yield inverse(oracle3(qq))

In [18]:
import cirq
from cirq import X, H, Z, inverse, CX, CCX

def grover3(trials_number):
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(13)
    n=4

    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation3(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [19]:
#You can use this cell to test your solution
shots=1000
grover3(shots)

Counter({'0010': 61,
         '0100': 73,
         '0001': 73,
         '1100': 65,
         '1011': 62,
         '1001': 62,
         '1010': 63,
         '0011': 65,
         '1110': 61,
         '1101': 54,
         '1000': 62,
         '1111': 64,
         '0101': 59,
         '0111': 52,
         '0110': 69,
         '0000': 55})

In [ ]:
# hidden tests in this cell will be used for grading.